# 15 · Falsos positivos en correo corporativo real (Enron ham) — evaluación fuera de distribución
**Por qué:** en la prueba del stack real (14/10) dos correos legítimos de estilo corporativo moderno salieron como phishing
(riesgo 19 y 70). El legítimo del corpus (CEAS, SpamAssassin) es sobre todo listas de correo técnicas: el FPR de 0,9 % del test
no representa el correo de oficina. Aquí se mide con **ham de Enron** (Enron-Spam, Metsis et al., 2006; copia `SetFit/enron_spam`
en Hugging Face), una fuente que el modelo nunca vio.

**Diseño honesto:** 2.000 ham Enron se parten en A (1.000, solo para elegir un umbral operativo alternativo) y B (1.000, solo para
reportar). El recall se mide sobre **todo el phishing del test held-out** en inglés. No se reentrena nada.
Runtime: GPU recomendada (T4: ~10 min; CPU: ~40 min).

In [1]:
# Celda 1: Entorno y detector
import os, sys, json, time, importlib, subprocess
from pathlib import Path
import numpy as np, pandas as pd
EN_COLAB = os.path.exists("/content")
if EN_COLAB:
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
else:
    BASE = next(Path("/kaggle/input").rglob("PI_G17"))
sys.path.insert(0, str(BASE / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "langdetect"], check=True)
import torch, inference
DISP = "cuda" if torch.cuda.is_available() else "cpu"
DET = inference.Detector.desde_directorio(BASE, dispositivo=DISP)
UMBRAL = DET.umbral
print("dispositivo:", DISP, "| umbral V3 (FPR ≤ 1 % en validación):", round(UMBRAL, 4))

Mounted at /content/drive


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

dispositivo: cuda | umbral V3 (FPR ≤ 1 % en validación): 0.039


In [2]:
# Celda 2: Datos: ham Enron (A/B) + test held-out inglés
from datasets import load_dataset
SEMILLA = 17
ds = load_dataset("SetFit/enron_spam")
H = pd.concat([ds[s].to_pandas() for s in ds.keys()])
H = H[H.label == 0].drop_duplicates("message")
H = H[H.message.fillna("").str.len().between(20, 20000)].sample(2000, random_state=SEMILLA).reset_index(drop=True)
H["mitad"] = np.where(np.arange(len(H)) < 1000, "A", "B")
V1 = pd.read_parquet(BASE / "data/processed/emails_v1.parquet", columns=["id", "hash", "clase", "split"])
V0 = pd.read_parquet(BASE / "data/interim/emails_v0.parquet", columns=["hash", "texto_defang"])
T = V1[V1.split == "test"].merge(V0.drop_duplicates("hash"), on="hash")
T = pd.concat([T[T.clase == "phishing"], T[T.clase == "legitimo"].sample(1000, random_state=SEMILLA)])
print("ham Enron:", len(H), "| test phishing:", (T.clase == "phishing").sum(), "| test legítimo:", (T.clase == "legitimo").sum())

README.md:   0%|          | 0.00/176 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B /  101MB            

train.jsonl: downloading bytes:           |  0.00B            

test.jsonl: reconstructing file:   0%|          |  0.00B / 6.27MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/31716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

ham Enron: 2000 | test phishing: 1488 | test legítimo: 1000


In [3]:
# Celda 3: Inferencia con el detector de producción
def correr(asunto, cuerpo):
    s = DET.analizar(asunto or "", cuerpo or "")
    return s["ruta_modelo"], s["clasificacion"]["probabilidades"]["phishing"], s["clasificacion"]["clase"]
t0 = time.time()
H[["ruta", "pph", "clase_pred"]] = pd.DataFrame([correr(a, c) for a, c in zip(H.subject, H.message)], index=H.index)
T = T.reset_index(drop=True)
T[["ruta", "pph", "clase_pred"]] = pd.DataFrame([correr("", c) for c in T.texto_defang], index=T.index)
print("minutos:", round((time.time() - t0) / 60, 1), "| rutas Enron:", H.ruta.value_counts().to_dict())

minutos: 4.0 | rutas Enron: {'en_v3': 2000}


In [4]:
# Celda 4: Métricas, umbral operativo elegido en A, reportado en B
def wilson(k, n, z=1.959964):
    if n == 0: return (0.0, 1.0)
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; m = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / d
    return (max(0.0, c - m), min(1.0, c + m))
He = H[H.ruta == "en_v3"]; A, B = He[He.mitad == "A"], He[He.mitad == "B"]
Tph, Tle = T[T.clase == "phishing"], T[T.clase == "legitimo"]
def fpr(d, u): return float((d.pph >= u).mean())
def ic(d, u):
    k = int((d.pph >= u).sum()); lo, hi = wilson(k, len(d)); return [round(100 * lo, 1), round(100 * hi, 1)]
def fila(u):
    return {"umbral": round(u, 4), "FPR_enron_B_pct": round(100 * fpr(B, u), 1), "IC95_B": ic(B, u),
            "FPR_test_legit_pct": round(100 * fpr(Tle, u), 2), "recall_test_phishing_pct": round(100 * (Tph.pph >= u).mean(), 2)}
GRID = [UMBRAL, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99]
TABLA = [fila(u) for u in GRID]
# Umbral operativo: el más bajo con FPR ≤ 2 % (meta SMART) en la mitad A; se reporta en B
cands = np.unique(np.r_[A.pph.values, Tph.pph.values])
ok = [u for u in cands if u >= UMBRAL and fpr(A, u) <= 0.02]
U_OP = float(min(ok)) if ok else None
corto = He.message.str.len() < 300
RESUMEN = {
    "n_enron_en": len(He), "n_test_phishing": len(Tph), "umbral_actual": round(UMBRAL, 4),
    "actual": fila(UMBRAL),
    "enron_ham_pred_spam_pct": round(100 * (He.clase_pred == "spam").mean(), 1),
    "FPR_enron_por_longitud": {"<300 car.": round(100 * fpr(He[corto], UMBRAL), 1), ">=300 car.": round(100 * fpr(He[~corto], UMBRAL), 1)},
    "umbral_operativo_A": None if U_OP is None else fila(U_OP),
    "tabla": TABLA}
(BASE / "results").mkdir(exist_ok=True)
(BASE / "results/fpr_enron_v1.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False))
print("===== RESUMEN (15) =====")
print(json.dumps(RESUMEN, ensure_ascii=False, separators=(",", ":")))

===== RESUMEN (15) =====
{"n_enron_en":2000,"n_test_phishing":1488,"umbral_actual":0.039,"actual":{"umbral":0.039,"FPR_enron_B_pct":12.5,"IC95_B":[10.6,14.7],"FPR_test_legit_pct":0.1,"recall_test_phishing_pct":99.73},"enron_ham_pred_spam_pct":2.5,"FPR_enron_por_longitud":{"<300 car.":20.3,">=300 car.":10.6},"umbral_operativo_A":{"umbral":0.8586,"FPR_enron_B_pct":1.3,"IC95_B":[0.8,2.2],"FPR_test_legit_pct":0.0,"recall_test_phishing_pct":95.97},"tabla":[{"umbral":0.039,"FPR_enron_B_pct":12.5,"IC95_B":[10.6,14.7],"FPR_test_legit_pct":0.1,"recall_test_phishing_pct":99.73},{"umbral":0.3,"FPR_enron_B_pct":6.0,"IC95_B":[4.7,7.6],"FPR_test_legit_pct":0.0,"recall_test_phishing_pct":99.46},{"umbral":0.5,"FPR_enron_B_pct":5.0,"IC95_B":[3.8,6.5],"FPR_test_legit_pct":0.0,"recall_test_phishing_pct":99.33},{"umbral":0.7,"FPR_enron_B_pct":3.1,"IC95_B":[2.2,4.4],"FPR_test_legit_pct":0.0,"recall_test_phishing_pct":98.79},{"umbral":0.8,"FPR_enron_B_pct":2.4,"IC95_B":[1.6,3.5],"FPR_test_legit_pct":0.0,"re